# Flavor

Flavor is an ordinary `IndexType` with `is_flavor=True` and a finite
`dimension`. A field that carries it becomes a class: `class_members` names
the generations, and `feynman_rule(..., flavor_expand=True)` unfolds them.


## Setup


In [ ]:
import re
import sys
from fractions import Fraction
from pathlib import Path

from symbolica import Expression, S

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
SRC = ROOT / "src"
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

ANSI_ESCAPE_RE = re.compile(r"\x1B\[[0-?]*[ -/]*[@-~]")


def clean(text):
    return ANSI_ESCAPE_RE.sub("", str(text))


def show(title, result):
    print("==========")
    print(title)
    if isinstance(result, dict):
        print(f"{len(result)} vertex signature(s)")
        print()
        for signature, expression in result.items():
            print("Vertex:", signature)
            print("Rule:", clean(expression))
            print()
    else:
        print(clean(result))
        print()


def show_model(model, *fields, compact_form=None, sum_notation=None, simplify=True):
    source_terms = model.lagrangian_decl.source_terms
    if source_terms:
        lagrangian_source = (
            sum(source_terms[1:], source_terms[0])
            if len(source_terms) > 1
            else source_terms[0]
        )
        show("Lagrangian", lagrangian_source)
    lagrangian = model.lagrangian()
    if fields:
        show("Feynman Rule", lagrangian.feynman_rule(*fields, include_delta=False, simplify=simplify))
    else:
        show("Feynman Rules", lagrangian.feynman_rule(include_delta=False, simplify=simplify))
    if compact_form is not None:
        show("Compact Form", compact_form)
    if sum_notation is not None:
        show("Sum Notation", sum_notation)

from feynpy import (
    COLOR_FUND_INDEX,
    Field,
    Model,
    Parameter,
    SPINOR_INDEX,
    flavor_index,
)


## Flavor classes

`l` is a three-generation lepton class. The compact rule keeps the class
field; the expanded rules are diagonal `e`, `mu`, `ta` vertices.


In [ ]:
Generation = flavor_index("Generation", 3, prefix="f")
f, h = S("f"), S("h")

l = Field(
    "l",
    spin=Fraction(1, 2),
    self_conjugate=False,
    indices=(Generation, SPINOR_INDEX),
    symbol=S("l"),
    conjugate_symbol=S("lbar"),
    flavor_index=Generation,
    class_members=("e", "mu", "ta"),
)
e, mu, ta = l.class_members
Phi = Field("Phi", spin=0, self_conjugate=True, symbol=S("Phi"))

show("Generation", Generation)
show("class members", [member.name for member in l.class_members])

lepton_model = Model(S("lam") * l.bar(f) * l(f) * Phi)
show_model(lepton_model, l.bar, l, Phi)
show("expanded rules", lepton_model.feynman_rule(flavor_expand=True, simplify=True))


Spectator indices such as color are inherited by every class member.


In [ ]:
fq, color = S("fq", "c")
q = Field(
    "q",
    spin=Fraction(1, 2),
    self_conjugate=False,
    indices=(Generation, COLOR_FUND_INDEX, SPINOR_INDEX),
    symbol=S("q"),
    conjugate_symbol=S("qbar"),
    flavor_index=Generation,
    class_members=("d", "s", "b"),
)
color_model = Model(S("gQ") * q.bar(fq, color) * q(fq, color) * Phi)
show_model(color_model)
show("expanded rules", color_model.feynman_rule(flavor_expand=True, simplify=True))


## Flavor tensors

A two-index parameter `Y(f, h)` generates off-diagonal vertices. Setting the
off-diagonal components to zero keeps only the diagonal family.


In [ ]:
Y = Parameter("Y", indices=(Generation, Generation))
mixed_model = Model(
    S("gY") * l.bar(f) * Y(f, h) * l(h) * Phi,
    parameters=(Y,),
)
show_model(mixed_model)
show("e.bar, mu, Phi", mixed_model.feynman_rule(e.bar, mu, Phi, simplify=True, flavor_expand=True))

Ydiag = Parameter(
    "Ydiag",
    indices=(Generation, Generation),
    components={
        (row, col): 0
        for row in range(1, Generation.dimension + 1)
        for col in range(1, Generation.dimension + 1)
        if row != col
    },
)
diag_model = Model(
    S("gDiag") * l.bar(f) * Ydiag(f, h) * l(h) * Phi,
    parameters=(Ydiag,),
)
show("diagonal signatures", [s.names for s in diag_model.vertex_signatures(flavor_expand=True)])
show("expanded diagonal rules", diag_model.feynman_rule(flavor_expand=True, simplify=True))


A one-index parameter `y(f)` in `y(f) * l.bar(f) * l(f)` is allowed by default.
Set `allow_summation=False` to reject that pattern.


In [ ]:
y_diag = Parameter("yDiag", indices=(Generation,))
sum_model = Model(y_diag(f) * l.bar(f) * l(f) * Phi, parameters=(y_diag,))
show("default summation", sum_model.feynman_rule(flavor_expand=True, simplify=True))

y_opt_out = Parameter("yOptOut", indices=(Generation,), allow_summation=False)
try:
    Model(y_opt_out(f) * l.bar(f) * l(f) * Phi, parameters=(y_opt_out,)).vertex_signatures(
        flavor_expand=True
    )
except Exception as exc:
    show("allow_summation=False", f"{type(exc).__name__}: {exc}")


## Shared and independent labels

The same generation label synchronizes two classes. Independent labels give
CKM-like mixing.


In [ ]:
lL = Field(
    "lL",
    spin=Fraction(1, 2),
    self_conjugate=False,
    indices=(Generation, SPINOR_INDEX),
    symbol=S("lL"),
    conjugate_symbol=S("lLbar"),
    flavor_index=Generation,
    class_members=("eL", "muL", "taL"),
)
lR = Field(
    "lR",
    spin=Fraction(1, 2),
    self_conjugate=False,
    indices=(Generation, SPINOR_INDEX),
    symbol=S("lR"),
    conjugate_symbol=S("lRbar"),
    flavor_index=Generation,
    class_members=("eR", "muR", "taR"),
)
yLR = Parameter("yLR", indices=(Generation,))
shared_model = Model(yLR(f) * lL.bar(f) * lR(f) * Phi, parameters=(yLR,))
show_model(shared_model)
show("expanded shared rules", shared_model.feynman_rule(flavor_expand=True, simplify=True))

uq = Field(
    "uq",
    spin=Fraction(1, 2),
    self_conjugate=False,
    indices=(Generation, COLOR_FUND_INDEX, SPINOR_INDEX),
    symbol=S("uq"),
    conjugate_symbol=S("uqbar"),
    flavor_index=Generation,
    class_members=("u", "c", "t"),
)
dq = Field(
    "dq",
    spin=Fraction(1, 2),
    self_conjugate=False,
    indices=(Generation, COLOR_FUND_INDEX, SPINOR_INDEX),
    symbol=S("dq"),
    conjugate_symbol=S("dqbar"),
    flavor_index=Generation,
    class_members=("d", "s", "b"),
)
W = Field("W", spin=0, self_conjugate=True, symbol=S("W"))
V = Parameter("V", indices=(Generation, Generation))
colour = S("colour")
mixing_model = Model(
    uq.bar(f, colour) * V(f, h) * dq(h, colour) * W,
    parameters=(V,),
)
show("CKM-like signatures", [s.names for s in mixing_model.vertex_signatures(flavor_expand=True)])
show("expanded mixing rules", mixing_model.feynman_rule(flavor_expand=True, simplify=True))


`flavor_expand` can target one index or several. The metadata layer fails early
when `class_members` does not match the flavor dimension.


In [ ]:
SU2D = flavor_index("SU2D", 2, prefix="d")
chi = Field(
    "chi",
    spin=Fraction(1, 2),
    self_conjugate=False,
    indices=(SU2D, SPINOR_INDEX),
    symbol=S("chi"),
    conjugate_symbol=S("chibar"),
    flavor_index=SU2D,
    class_members=("chi1", "chi2"),
)
d_sel, c_sel = S("d"), S("c")
selected_model = Model(S("gSel") * uq.bar(f, c_sel) * chi(d_sel) * Phi)
show("expand Generation only", [s.names for s in selected_model.vertex_signatures(flavor_expand=Generation)])
show("expand SU2D only", [s.names for s in selected_model.vertex_signatures(flavor_expand=SU2D)])
show("expand both", [s.names for s in selected_model.vertex_signatures(flavor_expand=(Generation, SU2D))])

try:
    Field(
        "BadPsi",
        spin=Fraction(1, 2),
        self_conjugate=False,
        indices=(Generation, SPINOR_INDEX),
        symbol=S("BadPsi"),
        conjugate_symbol=S("BadPsibar"),
        flavor_index=Generation,
        class_members=("e", "mu"),
    )
except Exception as exc:
    show("wrong member count", f"{type(exc).__name__}: {exc}")
